<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-10-tuning-and-evaluation/lesson-10.4-evaluation/practice/GCP_Capstone_10.4_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 10.4 — Vertex AI Evaluation

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: install, authenticate, init Vertex AI

Run this once. It installs the Vertex AI Evaluation SDK, authenticates with Application Default Credentials (Colab), and initialises Vertex AI with an experiment so every `EvalTask` run is tracked. The Evaluation service lives in `vertexai.evaluation` (a distinct managed service — not `vertexai.generative_models`), so it uses `vertexai.init` rather than a `genai.Client`.

In [ ]:
%%bash
pip install -q 'google-cloud-aiplatform[evaluation]' pandas

In [ ]:
# Application Default Credentials (runs only on Colab)
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab ADC')
except ImportError:
    print('Not on Colab - relying on local ADC (gcloud auth application-default login)')

import pandas as pd
import vertexai
from vertexai.evaluation import (
    EvalTask,
    PointwiseMetric,
    PairwiseMetric,
    PointwiseMetricPromptTemplate,
    MetricPromptTemplateExamples,
)

PROJECT = 'documind-ai-YOUR-ID'   # <-- your GCP project id
LOCATION = 'us-central1'          # course region (asia-south1 for India prod)
USD_INR = 85                      # for any INR cost display

vertexai.init(project=PROJECT, location=LOCATION, experiment='documind-eval-demo')
print('Vertex AI Evaluation SDK ready')

## Exercise 1: Build eval dataset

**Difficulty:** Easy

Create a pandas DataFrame with prompt + reference columns for 10 DocuMind document classifications.

**Expected behaviour:** DataFrame with 10 rows, columns `prompt` and `reference` (plus a pre-generated `response` column so computation metrics can run offline).

In [ ]:
# DocuMind document classification eval dataset (10 rows)
eval_df = pd.DataFrame({
    'prompt': [
        'Classify this document (INVOICE/CONTRACT/REPORT/RECEIPT):\nInvoice #INV-2024-001, Amount $4,590.00, Due 2024-04-14',
        'Classify this document:\nService Agreement between ACME and ClientX, Term 24 months effective 2024-01-01',
        'Classify this document:\nQ3 2024 Financial Report - Revenue grew 18% YoY to $47.2M',
        'Classify this document:\nReceipt #R-5521 from OfficeSupplies Inc, Date 2024-03-20, Total $114.50',
        'Classify this document:\nInvoice #INV-2024-002 for Cloud Services, $12,000, Net 30',
        'Classify this document:\nMaster Services Agreement, 36-month term, signed by both parties 2024-02-10',
        'Classify this document:\nAnnual Report FY2024 - net profit margin improved to 12.4%',
        'Classify this document:\nReceipt #R-5590 from CafeBlue, Date 2024-03-28, Total $18.75',
        'Classify this document:\nInvoice #INV-2024-003, Consulting, $8,250.00, Payment Terms Net 45',
        'Classify this document:\nNon-Disclosure Agreement between DocuMind Inc and VendorY, effective 2024-01-15',
    ],
    'reference': [
        'INVOICE', 'CONTRACT', 'REPORT', 'RECEIPT', 'INVOICE',
        'CONTRACT', 'REPORT', 'RECEIPT', 'INVOICE', 'CONTRACT',
    ],
    # Pre-generated responses to avoid live inference during the demo
    'response': [
        'INVOICE', 'CONTRACT', 'REPORT', 'RECEIPT', 'INVOICE',
        'CONTRACT', 'REPORT', 'RECEIPT', 'INVOICE', 'CONTRACT',
    ],
})

print(f'Dataset: {len(eval_df)} rows')
print(eval_df[['prompt', 'reference']].head())

## Exercise 2: Run exact_match

**Difficulty:** Easy

Build an `EvalTask` with `exact_match` on the classification dataset and print `summary_metrics`.

**Expected behaviour:** `exact_match/mean` = classification accuracy in the range 0.0-1.0.

In [ ]:
# Computation-based metrics on the pre-generated responses.
# No model= is passed, so EvalTask scores the existing 'response' column
# (exact_match / rouge_l_sum / bleu need no candidate generation).
eval_task_classification = EvalTask(
    dataset=eval_df,
    metrics=['exact_match', 'rouge_l_sum', 'bleu'],
    experiment='documind-classification-comp',
)

result = eval_task_classification.evaluate(experiment_run_name='responses-v1')

print('Summary metrics:')
for k, v in result.summary_metrics.items():
    print(f'  {k:<24} {v}')

# Per-row breakdown
result.metrics_table

## Exercise 3: rouge_l_sum on summaries

**Difficulty:** Easy

Evaluate 10 generated summaries against references with `rouge_l_sum`.

**Expected behaviour:** `rouge_l_sum/mean` reflects summary quality vs the reference summaries.

In [ ]:
# Summary-quality dataset: generated summary in 'response', gold summary in 'reference'
summary_df = pd.DataFrame({
    'prompt': [f'Summarize document D{i:02d} in one sentence.' for i in range(1, 11)],
    'reference': [
        'Invoice INV-2024-001 for $4,590 is due 2024-04-14 on Net 30 terms.',
        'A 24-month service agreement between ACME and ClientX begins 2024-01-01.',
        'Q3 2024 revenue grew 18% year over year to $47.2M.',
        'Receipt R-5521 from OfficeSupplies Inc totals $114.50 on 2024-03-20.',
        'Invoice INV-2024-002 bills $12,000 for cloud services on Net 30.',
        'A 36-month master services agreement was signed by both parties on 2024-02-10.',
        'FY2024 net profit margin improved to 12.4%.',
        'Receipt R-5590 from CafeBlue totals $18.75 on 2024-03-28.',
        'Invoice INV-2024-003 bills $8,250 for consulting on Net 45.',
        'An NDA between DocuMind Inc and VendorY is effective 2024-01-15.',
    ],
    'response': [
        'Invoice INV-2024-001 of $4,590 is due on 2024-04-14 under Net 30.',
        'ACME and ClientX signed a 24-month service agreement effective 2024-01-01.',
        'Revenue rose 18% YoY to $47.2M in Q3 2024.',
        'OfficeSupplies Inc receipt R-5521 came to $114.50 on 2024-03-20.',
        'Cloud services invoice INV-2024-002 is $12,000 on Net 30 terms.',
        'Both parties signed a 36-month master services agreement on 2024-02-10.',
        'Net profit margin rose to 12.4% in FY2024.',
        'CafeBlue receipt R-5590 was $18.75 on 2024-03-28.',
        'Consulting invoice INV-2024-003 is $8,250 on Net 45.',
        'DocuMind Inc and VendorY have an NDA effective 2024-01-15.',
    ],
})

summary_task = EvalTask(
    dataset=summary_df,
    metrics=['rouge_l_sum'],
    experiment='documind-summary-quality',
)
summary_result = summary_task.evaluate(experiment_run_name='summaries-v1')

print('rouge_l_sum/mean:', summary_result.summary_metrics.get('rouge_l_sum/mean'))
summary_result.metrics_table[['response', 'reference', 'rouge_l_sum/score']]

## Exercise 4: Gemini-as-judge groundedness

**Difficulty:** Medium

Evaluate RAG answers with the `groundedness` metric and review the judge's chain-of-thought explanations.

**Expected behaviour:** a groundedness score (1-5) per row, each with a written rationale.

In [ ]:
# RAG answer-quality dataset (context + response pre-populated)
rag_df = pd.DataFrame({
    'prompt': [
        'What are the payment terms for invoice INV-2024-001?',
        'What is the contract term?',
    ],
    'context': [
        'Invoice #INV-2024-001, Amount $4,590.00, Payment Terms: Net 30, Due 2024-04-14',
        'Service Agreement. Term: 24 months effective 2024-01-01.',
    ],
    'response': [
        'Payment terms are Net 30, with the invoice due on April 14, 2024.',
        'The contract term is 24 months, effective January 1, 2024.',
    ],
    'instruction': [
        'Answer the question using only the provided context.',
        'Answer the question using only the provided context.',
    ],
})

# Gemini-as-judge pointwise metrics (each returns 1-5 + chain-of-thought)
eval_task_rag = EvalTask(
    dataset=rag_df,
    metrics=[
        MetricPromptTemplateExamples.Pointwise.GROUNDEDNESS,
        MetricPromptTemplateExamples.Pointwise.FLUENCY,
        MetricPromptTemplateExamples.Pointwise.INSTRUCTION_FOLLOWING,
        MetricPromptTemplateExamples.Pointwise.QUESTION_ANSWERING_QUALITY,
    ],
    experiment='documind-rag-quality',
)

rag_result = eval_task_rag.evaluate(experiment_run_name='rag-judge-v1')

print('Groundedness mean:', rag_result.summary_metrics.get('groundedness/mean'))
# Per-row score + the judge's written rationale
rag_result.metrics_table[['response', 'groundedness/score', 'groundedness/explanation']]

## Exercise 5: Custom PointwiseMetric

**Difficulty:** Medium

Define `invoice_extraction_accuracy` with 3 criteria and a 5-point rubric.

**Expected behaviour:** a `PointwiseMetric` with `criteria` + `rating_rubric` that returns scores with explanations.

In [ ]:
# DocuMind invoice-extraction custom metric
extraction_accuracy = PointwiseMetric(
    metric='invoice_extraction_accuracy',
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            'Field Completeness': 'Does the extraction capture ALL required fields (invoice_id, vendor, date, total, line_items)?',
            'Value Accuracy':     'Are extracted values correct when compared against the source document?',
            'Format Compliance':  'Do extracted values use expected formats? Dates in ISO-8601 (YYYY-MM-DD), amounts as numbers (no currency symbols), line items as arrays.',
        },
        rating_rubric={
            '5': 'All fields correct, complete, properly formatted. Production-ready.',
            '4': 'Minor formatting issues (e.g., date format), all fields present and accurate.',
            '3': 'One field missing or one value incorrect.',
            '2': 'Multiple fields missing or multiple values incorrect.',
            '1': 'Extraction largely failed - most fields missing or wrong.',
        },
        input_variables=['prompt', 'response', 'reference'],
    ),
)

print('Custom metric:', extraction_accuracy.metric)
print('Criteria:', list(extraction_accuracy.metric_prompt_template.criteria.keys()))

In [ ]:
# Small extraction dataset to exercise the custom metric end-to-end
extraction_df = pd.DataFrame({
    'prompt': [
        'Extract all fields from: Invoice #INV-2024-001, Vendor ACME, Date 2024-03-15, Total $4,590.00, Items: [Consulting]',
        'Extract all fields from: Invoice #INV-2024-002, Vendor CloudCo, Date 03/20/2024, Total $12,000',
    ],
    'reference': [
        '{"invoice_id": "INV-2024-001", "vendor": "ACME", "date": "2024-03-15", "total": 4590.00, "line_items": ["Consulting"]}',
        '{"invoice_id": "INV-2024-002", "vendor": "CloudCo", "date": "2024-03-20", "total": 12000.00, "line_items": []}',
    ],
    'response': [
        '{"invoice_id": "INV-2024-001", "vendor": "ACME", "date": "2024-03-15", "total": 4590.00, "line_items": ["Consulting"]}',
        '{"invoice_id": "INV-2024-002", "vendor": "CloudCo", "date": "03/20/2024", "total": "$12,000"}',
    ],
})

extraction_task = EvalTask(
    dataset=extraction_df,
    metrics=[extraction_accuracy],
    experiment='documind-extraction-custom',
)
extraction_result = extraction_task.evaluate(experiment_run_name='extraction-v1')

extraction_result.metrics_table[[
    'response',
    'invoice_extraction_accuracy/score',
    'invoice_extraction_accuracy/explanation',
]]

## Exercise 6: Pairwise LoRA vs base

**Difficulty:** Medium

Build a `PairwiseMetric` with `baseline_model` = base Flash and compute the win rate.

**Expected behaviour:** `candidate_model_win_rate > 0.55` means the LoRA-tuned model beats the base model.

> In production the candidate is your LoRA-tuned endpoint from Lesson 10.1. To keep this notebook runnable end-to-end, the candidate below is a real named model; swap it for your tuned endpoint once you have one. Pairwise judging needs ~400 examples for statistical confidence — this small run is illustrative only.

In [ ]:
def build_pairwise_task(eval_df, baseline_model_name='gemini-3.6-flash'):
    '''Returns an EvalTask configured for pairwise comparison against a baseline.'''
    pairwise_qa = PairwiseMetric(
        metric='pairwise_question_answering_quality',
        metric_prompt_template=MetricPromptTemplateExamples.get_prompt_template(
            'pairwise_question_answering_quality'),
        baseline_model=baseline_model_name,
    )
    return EvalTask(
        dataset=eval_df,
        metrics=[
            pairwise_qa,
            'pairwise_groundedness',
            'pairwise_instruction_following',
        ],
        experiment='documind-lora-vs-base',
    )

# Baseline = base Flash; candidate = the model under test (your LoRA endpoint in prod)
pairwise_task = build_pairwise_task(rag_df, baseline_model_name='gemini-3.1-flash-lite')
CANDIDATE_MODEL = 'gemini-3.6-flash'  # replace with your LoRA-tuned endpoint resource name

pairwise_result = pairwise_task.evaluate(
    model=CANDIDATE_MODEL,
    experiment_run_name='lora-vs-base-v1',
)

win_rate = pairwise_result.summary_metrics.get(
    'pairwise_question_answering_quality/candidate_model_win_rate')
print(f'candidate_model_win_rate = {win_rate}')
print('Ship candidate' if (win_rate or 0) > 0.55 else 'Do not ship - candidate did not beat baseline')

## Exercise 7: Trajectory evaluation

**Difficulty:** Challenge

Evaluate the DocuMind agent on 5 queries with `trajectory_in_order_match`, `trajectory_precision`, and `trajectory_recall`.

**Expected behaviour:** per-row binary and float scores showing tool-sequence correctness.

In [ ]:
# DocuMind agent tool-calling evaluation over 5 queries
agent_df = pd.DataFrame({
    'prompt': [
        'Find invoices from Vendor X and summarize payment terms',
        'Extract vendor and total from the latest invoice',
        'List all contracts and count them',
        'Find the highest-value receipt and show its total',
        'Summarize the Q3 report',
    ],
    'reference_trajectory': [
        [
            {'tool_name': 'search_documents', 'tool_input': {'vendor': 'X', 'doc_type': 'INVOICE'}},
            {'tool_name': 'extract_fields', 'tool_input': {'fields': ['payment_terms']}},
            {'tool_name': 'summarize', 'tool_input': {'max_sentences': 3}},
        ],
        [
            {'tool_name': 'search_documents', 'tool_input': {'doc_type': 'INVOICE', 'limit': 1}},
            {'tool_name': 'extract_fields', 'tool_input': {'fields': ['vendor', 'total']}},
        ],
        [
            {'tool_name': 'search_documents', 'tool_input': {'doc_type': 'CONTRACT'}},
            {'tool_name': 'count', 'tool_input': {}},
        ],
        [
            {'tool_name': 'search_documents', 'tool_input': {'doc_type': 'RECEIPT'}},
            {'tool_name': 'extract_fields', 'tool_input': {'fields': ['total']}},
            {'tool_name': 'rank', 'tool_input': {'by': 'total', 'order': 'desc'}},
        ],
        [
            {'tool_name': 'search_documents', 'tool_input': {'doc_type': 'REPORT', 'query': 'Q3'}},
            {'tool_name': 'summarize', 'tool_input': {'max_sentences': 3}},
        ],
    ],
    'predicted_trajectory': [
        [
            {'tool_name': 'search_documents', 'tool_input': {'vendor': 'X', 'doc_type': 'INVOICE'}},
            {'tool_name': 'extract_fields', 'tool_input': {'fields': ['payment_terms']}},
            {'tool_name': 'summarize', 'tool_input': {'max_sentences': 3}},
        ],
        [
            {'tool_name': 'search_documents', 'tool_input': {'doc_type': 'INVOICE'}},
            {'tool_name': 'extract_fields', 'tool_input': {'fields': ['vendor', 'total', 'date']}},  # extra field
        ],
        [
            {'tool_name': 'search_documents', 'tool_input': {'doc_type': 'CONTRACT'}},
            {'tool_name': 'count', 'tool_input': {}},
        ],
        [
            {'tool_name': 'search_documents', 'tool_input': {'doc_type': 'RECEIPT'}},
            {'tool_name': 'rank', 'tool_input': {'by': 'total', 'order': 'desc'}},  # skipped extract_fields
        ],
        [
            {'tool_name': 'summarize', 'tool_input': {'max_sentences': 3}},  # wrong order - summarize before search
            {'tool_name': 'search_documents', 'tool_input': {'doc_type': 'REPORT', 'query': 'Q3'}},
        ],
    ],
})

def trajectory_exact_match(pred, ref):
    return int(pred == ref)

def trajectory_in_order_match(pred, ref):
    pred_names = [t['tool_name'] for t in pred]
    ref_names = [t['tool_name'] for t in ref]
    i = 0
    for name in pred_names:
        if i < len(ref_names) and name == ref_names[i]:
            i += 1
    return int(i == len(ref_names))

def trajectory_any_order_match(pred, ref):
    pred_names = set(t['tool_name'] for t in pred)
    ref_names = set(t['tool_name'] for t in ref)
    return int(ref_names.issubset(pred_names))

def trajectory_precision(pred, ref):
    if not pred:
        return 0.0
    ref_names = set(t['tool_name'] for t in ref)
    relevant = sum(1 for t in pred if t['tool_name'] in ref_names)
    return relevant / len(pred)

def trajectory_recall(pred, ref):
    if not ref:
        return 0.0
    pred_names = set(t['tool_name'] for t in pred)
    captured = sum(1 for t in ref if t['tool_name'] in pred_names)
    return captured / len(ref)

metrics = [
    ('trajectory_exact_match', trajectory_exact_match),
    ('trajectory_in_order_match', trajectory_in_order_match),
    ('trajectory_any_order_match', trajectory_any_order_match),
    ('trajectory_precision', trajectory_precision),
    ('trajectory_recall', trajectory_recall),
]

header = 'Metric'.ljust(30) + ''.join(f'Row{i+1:>7}' for i in range(len(agent_df)))
print(header)
print('-' * len(header))
for name, fn in metrics:
    row = name.ljust(30)
    for i in range(len(agent_df)):
        score = fn(agent_df['predicted_trajectory'][i], agent_df['reference_trajectory'][i])
        row += f'{score:>7.2f}'
    print(row)

## Exercise 8: Experiments tracking pipeline

**Difficulty:** Challenge

Compare 4 prompt variations in one experiment, then render a radar plot via `notebook_utils`.

**Expected behaviour:** 4 `experiment_run_name` entries in the same experiment, with a radar-plot comparison across metrics.

In [ ]:
# Systematic prompt comparison - all runs land in one experiment
prompt_templates = [
    'Classify the document (INVOICE/CONTRACT/REPORT/RECEIPT): {prompt}',
    'You are DocuMind. Return the category name only. Document: {prompt}',
    'Task: classify. Options: INVOICE, CONTRACT, REPORT, RECEIPT.\nDocument: {prompt}\nCategory:',
    'Read the document and reply with exactly one of INVOICE, CONTRACT, REPORT, RECEIPT.\n{prompt}',
]

eval_task_prompts = EvalTask(
    dataset=eval_df,
    metrics=['exact_match', 'rouge_l_sum'],
    experiment='documind-prompt-comparison',
)

runs = []
for idx, template in enumerate(prompt_templates):
    result = eval_task_prompts.evaluate(
        model='gemini-3.6-flash',
        prompt_template=template,
        experiment_run_name=f'prompt-{idx}',
    )
    runs.append((f'prompt-{idx}', result))
    print(f'prompt-{idx}: exact_match/mean =', result.summary_metrics.get('exact_match/mean'))

print('\nAll 4 runs tracked under experiment "documind-prompt-comparison"')
print('View in Console: Vertex AI -> Model Development -> Experiments')

In [ ]:
# Radar-plot comparison across the 4 prompt runs
from vertexai.preview.evaluation import notebook_utils

notebook_utils.display_radar_plot(
    runs,
    metrics=['exact_match', 'rouge_l_sum'],
)